# codingStandard — Colab clean-runtime validation

Run this notebook top-to-bottom in a fresh Google Colab runtime. The notebook clones the public distribution repository automatically, so the current working directory does not need to be the repository root.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/eaglesjo/codingStandard.git'
REPO_DIR = Path('/content/codingStandard')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print('Repository:', REPO_URL)
print('CWD:', Path.cwd())
print('Python:', sys.version)

In [ ]:
%pip install -q psutil
print('psutil ready')

In [ ]:
result = subprocess.run([sys.executable, 'platform/colab/validate_runtime.py', '--json'], check=True, capture_output=True, text=True)
report = json.loads(result.stdout)
print(json.dumps(report, indent=2, sort_keys=True))
assert report['is_colab'] is True
assert report['os'] == 'Linux'

In [ ]:
subprocess.run([sys.executable, 'scripts/validation/validate_agent_routing.py'], check=True)
print('Agent routing validation passed')

In [ ]:
import importlib.util
if importlib.util.find_spec('torch') is None:
    print('PyTorch is not installed; skipping framework smoke test.')
else:
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    x = torch.randn(4, 8, device=device)
    layer = torch.nn.Linear(8, 2, device=device)
    loss = layer(x).square().mean()
    loss.backward()
    print('PyTorch smoke passed:', device, 'torch', torch.__version__)

In [ ]:
if importlib.util.find_spec('torch') is None:
    print('PyTorch unavailable; checkpoint restore test skipped.')
else:
    import torch
    durable = Path('/content/codingstandard-validation')
    durable.mkdir(parents=True, exist_ok=True)
    path = durable / 'checkpoint.pt'
    state = {'step': 1, 'tensor': torch.arange(4)}
    torch.save(state, path)
    restored = torch.load(path, map_location='cpu', weights_only=False)
    assert restored['step'] == 1
    assert restored['tensor'].tolist() == [0, 1, 2, 3]
    print('checkpoint/restore passed:', path)

In [ ]:
summary = {'repository': REPO_URL, 'repository_dir': str(REPO_DIR), 'runtime': report, 'checkpoint_path': '/content/codingstandard-validation/checkpoint.pt'}
print(json.dumps(summary, indent=2, sort_keys=True))
print('Clean runtime Colab validation passed')